<a href="https://colab.research.google.com/github/brindatenn/phd-prep-summer-2026/blob/main/week1/autograd_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to `torch.autograd`

**`torch.autograd`** is PyTorch's automatic differentiation engine. It's what powers neural network training by computing gradients for you.

**The big idea — training happens in two steps:**
- **Forward propagation:** the network runs the input through its functions to make a guess.
- **Backward propagation (backprop):** the network measures the error, then works *backwards*
  from the output collecting the derivative of the error w.r.t. each parameter (the **gradients**),
  and nudges the parameters to reduce the error (**gradient descent**).

Autograd's job is that backward step: given the operations you ran, it figures out every gradient automatically.

## 1. One training step in PyTorch

The whole training loop, in miniature: **forward → loss → `.backward()` → `optimizer.step()`**.

Here we use a pretrained `resnet18` and one fake image just to see the mechanics.

> ⚠️ Running this downloads the resnet18 weights (~45 MB) the first time. This example runs on **CPU**.

In [1]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Fake single image: (batch=1, channels=3, height=64, width=64) and a random target label
data   = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s]


In [2]:
# 1) FORWARD PASS - run the input through the model to get a prediction
prediction = model(data)

# 2) LOSS - how wrong the prediction is (toy loss here: just the summed error)
loss = (prediction - labels).sum()

# 3) BACKWARD PASS - autograd computes d(loss)/d(param) for every parameter,
#    and stores each result in that parameter's .grad attribute
loss.backward()

# 4) OPTIMIZER - SGD updates each parameter using its .grad
optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
optim.step()   # gradient descent: one step of adjusting the weights

That's a full training step. Everything below explains *how* `.backward()` actually works. (useful to truly understand)

## 2. How autograd collects gradients

Set **`requires_grad=True`** on a tensor to tell autograd: "track every operation on this,
because I'll want its gradient later."

We'll make two tensors `a`, `b` (pretend they're NN parameters) and build `Q` from them (pretend it's the error):

$$Q = 3a^3 - b^2$$

By hand, the gradients are:

$$\frac{\partial Q}{\partial a} = 9a^2 \qquad \frac{\partial Q}{\partial b} = -2b$$

We'll check autograd gets the same answers.

In [3]:
import torch

a = torch.tensor([2., 3.], requires_grad=True)   # track operations on a
b = torch.tensor([6., 4.], requires_grad=True)   # track operations on b

Q = 3*a**3 - b**2

`Q` is a **vector**, not a single number. When you call `.backward()` on a non-scalar you must pass a
`gradient` argument the same shape as `Q` - it represents dQ/dQ (a vector of 1s here).

> Simpler alternative: aggregate to a scalar first, then no argument is needed - `Q.sum().backward()`.
> (In real training the loss is already a scalar, so you just write `loss.backward()`.)

In [4]:
external_grad = torch.tensor([1., 1.])   # dQ/dQ, same shape as Q
Q.backward(gradient=external_grad)
# Equivalent shortcut: Q.sum().backward()

In [5]:
# Gradients are now stored in a.grad and b.grad - check they match the hand-derived formulas
print("dQ/da correct?", (9*a**2 == a.grad).all().item())    # 9a^2
print("dQ/db correct?", (-2*b == b.grad).all().item())      # -2b
print("a.grad:", a.grad)
print("b.grad:", b.grad)

dQ/da correct? True
dQ/db correct? True
a.grad: tensor([36., 81.])
b.grad: tensor([-12.,  -8.])


**Optional - the maths under the hood (vector-Jacobian product).**
For a vector function **y** = f(**x**), the matrix of all partial derivatives is the **Jacobian** J.
Autograd never builds J explicitly; it computes a **vector-Jacobian product** Jᵀ·**v**. When **v** is the
gradient of a scalar loss `l` (as it is in training), the chain rule makes Jᵀ·**v** exactly the gradient of
`l` w.r.t. **x**. That `external_grad` above *is* **v**. You don't need this to train models. Just know autograd is doing efficient chain-rule bookkeeping, not storing giant matrices.

## 3. The computational graph (DAG)

Autograd records tensors and the operations between them as a **directed acyclic graph (DAG)**:
- **Leaves** = your input tensors.
- **Roots** = the output tensors.
- Each node stores the operation's **gradient function** (`.grad_fn`).

**Forward pass** - autograd does two things at once: runs the operation to get the result,
*and* records that operation's gradient function in the DAG.

**Backward pass** (kicked off by `.backward()` on the root) - autograd:
1. reads the gradient from each `.grad_fn`,
2. accumulates it into each tensor's `.grad`,
3. chain-rules all the way back to the leaves.

> **The DAG is dynamic.** It's rebuilt from scratch after every `.backward()`. That's why you can
> use normal Python control flow (if/for) in a model and change shapes/operations each iteration.

In [6]:
# You can see the graph is being tracked via .grad_fn on non-leaf tensors
a = torch.tensor([2., 3.], requires_grad=True)
Q = 3*a**3
print("Q.grad_fn:", Q.grad_fn)          # points to the backward function of the last op
print("a is a leaf:", a.is_leaf)        # True - leaves are what gradients accumulate into

Q.grad_fn: <MulBackward0 object at 0x7ea8ad24d1e0>
a is a leaf: True


## 4. Excluding tensors: `requires_grad=False` & frozen parameters

Autograd only tracks tensors with `requires_grad=True`. Key rule:

> An operation's output requires gradients if **at least one** input requires gradients.

In [7]:
x = torch.rand(5, 5)                       # requires_grad defaults to False
y = torch.rand(5, 5)
z = torch.rand((5, 5), requires_grad=True)

a = x + y
print("a requires grad?", a.requires_grad)  # False: neither input tracks grads

b = x + z
print("b requires grad?", b.requires_grad)  # True: z tracks grads, so output does too

a requires grad? False
b requires grad? True


Parameters that don't compute gradients are called **frozen parameters**. Freezing is useful when you
know you won't need certain gradients. It saves computation.

**Finetuning** is the classic use: freeze the whole pretrained network, then replace/train only the
final classifier layer for your new task.

In [8]:
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze every parameter in the network
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier (resnet's final layer is model.fc) with a fresh one for 10 classes.
# New layers require grad by default, so ONLY these weights/bias will train.
model.fc = nn.Linear(512, 10)

# Even though we hand ALL params to the optimizer, only the unfrozen model.fc ones actually update
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

The same "don't track gradients" behaviour is available as a context manager, `torch.no_grad()` handy for inference/evaluation where you don't need gradients and want the speed/memory saving:

```python
with torch.no_grad():
    preds = model(data)   # no graph built, no gradients tracked
```

## Quick recap

- Training = **forward → loss → `.backward()` → `optimizer.step()`**; autograd handles the backward part.
- `requires_grad=True` tells autograd to **track operations** on a tensor.
- `.backward()` fills each leaf tensor's **`.grad`**; on a non-scalar you must pass a `gradient` arg
  (or call `.sum().backward()`). A scalar loss just needs `loss.backward()`.
- Autograd builds a **dynamic DAG** each pass and applies the chain rule from roots → leaves.
- An op's output requires grad if **any** input does; **freeze** params with `requires_grad=False`
  (or wrap inference in **`torch.no_grad()`**) to skip gradient work - the basis of finetuning.